[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/dataset_summary_cli.ipynb)

# Dataset Summary and CLI Utilities

This notebook demonstrates the new local dataset summary and subsetting helpers. It uses a small synthetic hyperspectral cube so the example runs without downloading external data.

In [ ]:
# %pip install hypercoast

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import xarray as xr

import hypercoast

## Create a small local cube

In [ ]:
wavelengths = np.array([450.0, 550.0, 650.0, 750.0], dtype="float32")
cube = xr.Dataset(
    {
        "reflectance": (
            ("wavelength", "y", "x"),
            np.arange(4 * 5 * 6, dtype="float32").reshape(4, 5, 6) / 100.0,
        )
    },
    coords={
        "wavelength": wavelengths,
        "y": np.linspace(28.0, 29.0, 5),
        "x": np.linspace(-91.0, -90.0, 6),
    },
    attrs={"crs": "EPSG:4326"},
)
cube

## Summarize metadata

In [ ]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "synthetic_cube.nc"
    cube.to_netcdf(path)

    summary = hypercoast.summarize_dataset(path, variable="reflectance")
    summary.as_dict()

The same metadata is available from the command line:

```bash
hypercoast summarize synthetic_cube.nc --variable reflectance --json
```

## Subset by bounding box

In [ ]:
with TemporaryDirectory() as tmpdir:
    input_path = Path(tmpdir) / "synthetic_cube.nc"
    output_path = Path(tmpdir) / "subset.nc"
    cube.to_netcdf(input_path)

    hypercoast.subset_dataset(
        input_path,
        output_path,
        bbox=(-90.8, 28.25, -90.2, 28.75),
        variable="reflectance",
    )
    subset = xr.open_dataset(output_path)
    subset

The equivalent CLI command is:

```bash
hypercoast subset synthetic_cube.nc subset.nc --bbox -90.8 28.25 -90.2 28.75 --variable reflectance
```

## Batch spectra command shape

For sensor readers that provide point extraction, the `spectra` command writes long-form spectral rows from a point CSV:

```bash
hypercoast spectra pace PACE_scene.nc --points stations.csv --output spectra.csv --x-column lon --y-column lat --crs EPSG:4326
```

The output columns are `feature_id`, `x`, `y`, `crs`, `wavelength`, `value`, `layer`, and `variable`.